# 01 — Dane

**Cel notebooka:** pobrać z [OpenF1](https://openf1.org/docs) wszystko, czego potrzebujemy do przewidzenia wyniku GP Stanów Zjednoczonych (Austin) 2026, i zapisać to na dysk — kolejne notebooki tylko wczytują pliki CSV.

> Pierwsze uruchomienie trwa kilka minut (limit zapytań API). Kolejne — sekundy, bo wszystko czytamy z `data/raw`.

## 1.1 Funkcja do pobierania danych

**Co robimy:** jedną funkcję `get()`, przez którą przechodzi każde zapytanie do OpenF1.

**Dlaczego:** będziemy pobierać dane z ponad stu sesji. Bez zapisu na dysk każde ponowne uruchomienie notebooka ściągałoby wszystko od nowa — wolno i obciążając API. Funkcja przy pierwszym wywołaniu pobiera dane i zapisuje JSON, przy kolejnych czyta z pliku.

In [1]:
import json, time
from pathlib import Path
import requests
import pandas as pd

BASE_URL = "https://api.openf1.org/v1"
CACHE_DIR = Path("data/raw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def get(endpoint, **params):
    """Pobiera dane z OpenF1; odpowiedź zapisuje na dysk i przy kolejnym wywołaniu czyta z pliku."""
    key = endpoint + "_" + "_".join(f"{k}={v}" for k, v in sorted(params.items()))
    path = CACHE_DIR / f"{key}.json"
    if path.exists():
        return pd.DataFrame(json.loads(path.read_text()))
    for attempt in range(5):
        resp = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)
        if resp.status_code != 429:
            break
        wait = 5 * 2 ** attempt
        print(f"429 (za dużo zapytań) — czekam {wait} s...")
        time.sleep(wait)
    time.sleep(1)
    if resp.status_code == 404:  # OpenF1 zwraca 404 "No results found", gdy danych (jeszcze) nie ma
        return pd.DataFrame()
    resp.raise_for_status()
    data = resp.json()
    path.write_text(json.dumps(data))
    return pd.DataFrame(data)

## 1.2 Kalendarz sezonu 2026

**Co robimy:** pobieramy wszystkie sesje 2026, zamieniamy daty z tekstu na daty i wyrzucamy odwołane sesje.

**Dlaczego:** z tej tabeli wiemy, które sesje już się odbyły, i bierzemy z niej `session_key` / `meeting_key` do kolejnych zapytań. Tekstu nie da się porównać z „teraz”, daty — tak.

In [2]:
sessions = get("sessions", year=2026)
for col in ["date_start", "date_end"]:
    sessions[col] = pd.to_datetime(sessions[col])
sessions = sessions[~sessions["is_cancelled"]]

sessions[sessions["country_name"] == "United States"][["session_key", "session_name", "location", "date_start"]]

,session_key,session_name,location,date_start
31,11270,Practice 1,Miami Gardens,2026-05-01 16:00:00+00:00
32,11271,Sprint Qualifying,Miami Gardens,2026-05-01 20:30:00+00:00
33,11275,Sprint,Miami Gardens,2026-05-02 16:00:00+00:00
34,11276,Qualifying,Miami Gardens,2026-05-02 20:00:00+00:00
35,11280,Race,Miami Gardens,2026-05-03 17:00:00+00:00
101,11389,Practice 1,Austin,2026-10-23 17:30:00+00:00
102,11390,Practice 2,Austin,2026-10-23 21:00:00+00:00
103,11391,Practice 3,Austin,2026-10-24 17:30:00+00:00
104,11392,Qualifying,Austin,2026-10-24 21:00:00+00:00
105,11396,Race,Austin,2026-10-25 20:00:00+00:00


## 1.3 Wyścigi: zakończone + dzisiejszy

**Co robimy:** bierzemy wyścigi, które już wystartowały albo startują dziś, numerujemy je jako rundy i oznaczamy, które są zakończone.

**Dlaczego:** zakończone wyścigi to zbiór treningowy (znamy wynik). Dzisiejszy — Austin — to ten, dla którego robimy predykcję: znamy jego kwalifikacje (i sprint, jeśli był), ale nie wynik. Numer rundy przyda się do liczenia formy („ostatnie 3 wyścigi”).

In [3]:
now = pd.Timestamp.now(tz="UTC")
races = (
    sessions[(sessions["session_name"] == "Race") & (sessions["date_start"] < now + pd.Timedelta(days=1))]
    .sort_values("date_start")
    .reset_index(drop=True)
)
races["round"] = races.index + 1
races["finished"] = races["date_end"] < now
races_done = races[races["finished"]]

TARGET_MEETING = races.loc[races["location"] == "Austin", "meeting_key"].item()

print(len(races_done), "zakończonych wyścigów; cel predykcji: meeting_key =", TARGET_MEETING)
races[["round", "session_key", "meeting_key", "country_name", "location", "date_start", "finished"]]

17 zakończonych wyścigów; cel predykcji: meeting_key = 1297


,round,session_key,meeting_key,country_name,location,date_start,finished
0,1,11234,1279,Australia,Melbourne,2026-03-08 04:00:00+00:00,True
1,2,11245,1280,China,Shanghai,2026-03-15 07:00:00+00:00,True
2,3,11253,1281,Japan,Suzuka,2026-03-29 05:00:00+00:00,True
3,4,11280,1284,United States,Miami Gardens,2026-05-03 17:00:00+00:00,True
4,5,11291,1285,Canada,Montréal,2026-05-24 20:00:00+00:00,True
5,6,11299,1286,Monaco,Monte Carlo,2026-06-07 13:00:00+00:00,True
6,7,11307,1287,Spain,Barcelona,2026-06-14 13:00:00+00:00,True
7,8,11315,1288,Austria,Spielberg,2026-06-28 13:00:00+00:00,True
8,9,11326,1289,United Kingdom,Silverstone,2026-07-05 14:00:00+00:00,True
9,10,11334,1290,Belgium,Spa-Francorchamps,2026-07-19 13:00:00+00:00,True


## 1.4 Kierowcy weekendu

**Co robimy:** funkcję, która dla danego weekendu (`meeting_key`) zwraca tabelę: numer → skrót nazwiska → zespół.

**Dlaczego:** wszystkie wyniki w OpenF1 znają kierowcę tylko po `driver_number`, a numer **nie jest stałym identyfikatorem** (w 2026 Norris jeździ z #1, Verstappen z #3). Do łączenia danych między sezonami używamy `name_acronym` (VER, NOR…), a `team_name` jest potrzebny do porównań z partnerem z zespołu. Pytamy po `meeting_key` (1 zapytanie na weekend) zamiast po każdej sesji osobno — mniej zapytań, mniej problemów z limitem.

In [4]:
DRIVER_COLS = ["driver_number", "name_acronym", "full_name", "team_name"]

def weekend_drivers(meeting_key):
    """Kierowcy weekendu: driver_number -> name_acronym, team_name."""
    drv = get("drivers", meeting_key=meeting_key)
    return drv[DRIVER_COLS].drop_duplicates("driver_number")

def session_key(meeting_key, session_name):
    """session_key konkretnej sesji weekendu albo None, jeśli takiej sesji nie było (np. sprint)."""
    s = sessions[(sessions["meeting_key"] == meeting_key) & (sessions["session_name"] == session_name)]
    return s["session_key"].item() if len(s) else None

weekend_drivers(TARGET_MEETING)

,driver_number,name_acronym,full_name,team_name
0,1,NOR,Lando NORRIS,McLaren
1,3,VER,Max VERSTAPPEN,Red Bull Racing
2,5,BOR,Gabriel BORTOLETO,Audi
3,6,HAD,Isack HADJAR,Red Bull Racing
4,10,GAS,Pierre GASLY,Alpine
5,12,ANT,Kimi ANTONELLI,Mercedes
6,14,ALO,Fernando ALONSO,Aston Martin
7,16,LEC,Charles LECLERC,Ferrari
8,18,STR,Lance STROLL,Aston Martin
9,23,ALB,Alexander ALBON,Williams


## 1.5 Wyniki wyścigów

**Co robimy:** dla każdego zakończonego wyścigu pobieramy `session_result` i doklejamy kierowców.

**Dlaczego:** to nasze **zmienne docelowe** — z pozycji na mecie policzymy później „wygrał / top 3 / top 10”. A historia wyników posłuży też do cech formy.

In [5]:
frames = []
for race in races_done.itertuples():
    res = get("session_result", session_key=race.session_key)
    df = res.merge(weekend_drivers(race.meeting_key), on="driver_number", how="left")
    df["round"] = race.round
    df["meeting_key"] = race.meeting_key
    frames.append(df)

results = pd.concat(frames, ignore_index=True)
print(results.shape)
results.head()

(374, 15)


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,session_key,name_acronym,full_name,team_name,round
0,1.0,63,58,25.0,False,False,False,4986.801,0,1279,11234,RUS,George RUSSELL,Mercedes,1
1,2.0,12,58,18.0,False,False,False,4989.775,2.974,1279,11234,ANT,Kimi ANTONELLI,Mercedes,1
2,3.0,16,58,15.0,False,False,False,5002.320,15.519,1279,11234,LEC,Charles LECLERC,Ferrari,1
3,4.0,44,58,12.0,False,False,False,5002.945,16.144,1279,11234,HAM,Lewis HAMILTON,Ferrari,1
4,5.0,1,58,10.0,False,False,False,5038.542,51.741,1279,11234,NOR,Lando NORRIS,McLaren,1


### Kontrola jakości

**Dlaczego:** błędy w danych przechodzą po cichu przez cały pipeline i wychodzą dopiero jako dziwne predykcje. Taniej złapać je teraz.

In [6]:
print("Brak dopasowania kierowcy:", results["name_acronym"].isna().sum())
print("Kierowców na wyścig:", results.groupby("round").size().unique())
print()
print(results[["dnf", "dns", "dsq"]].sum())
results[results["dnf"]][["round", "name_acronym", "team_name", "position", "number_of_laps"]].head(10)

Brak dopasowania kierowcy: 0
Kierowców na wyścig: [22]

dnf    71
dns     7
dsq     0
dtype: int64


,round,name_acronym,team_name,position,number_of_laps
17,1,ALO,Aston Martin,NaN,21
18,1,BOT,Cadillac,NaN,15
19,1,HAD,Red Bull Racing,NaN,10
37,2,VER,Red Bull Racing,NaN,45
38,2,ALO,Aston Martin,NaN,32
39,2,STR,Aston Martin,NaN,9
64,3,STR,Aston Martin,NaN,30
65,3,BEA,Haas F1 Team,NaN,20
84,4,HUL,Audi,NaN,7
85,4,LAW,Racing Bulls,NaN,6


## 1.6 Kwalifikacje i sprinty (także Austin)

**Co robimy:** dla każdego weekendu — łącznie z dzisiejszym — pobieramy wyniki kwalifikacji i sprintu.

**Dlaczego:**
- **Kwalifikacje** = pozycja startowa (najważniejsza pojedyncza cecha) + czyste tempo samochodu i kierowcy.
- Endpoint `starting_grid` jest w 2026 pusty, więc **pozycja w kwalifikacjach zastępuje pozycję startową**. Tracimy informację o karach na starcie — świadome uproszczenie.
- **Sprint** — dodatkowa informacja o tempie wyścigowym w tym samym weekendzie (tylko na weekendach sprinterskich).

W kwalifikacjach `duration` to **lista trzech czasów `[Q1, Q2, Q3]`** (z `None` w segmentach, w których kierowca już nie jechał). Liczymy z niej:
- `best_time` — najlepszy czas kierowcy,
- `gap_to_pole_pct` — o ile procent wolniej od najszybszego. Procenty zamiast sekund, bo okrążenie w Monako i na Spa ma zupełnie inną długość.

In [7]:
def best_time(durations):
    """Najlepszy czas z [Q1, Q2, Q3]; None, gdy kierowca nie ustanowił żadnego czasu."""
    if not isinstance(durations, list):
        return None
    times = [t for t in durations if t is not None]
    return min(times) if times else None

def load_weekend_session(session_name):
    """Wyniki danej sesji (np. 'Qualifying') ze wszystkich weekendów w `races`, z doklejonymi kierowcami."""
    frames = []
    for race in races.itertuples():
        key = session_key(race.meeting_key, session_name)
        if key is None:
            continue
        res = get("session_result", session_key=key)
        if res.empty:
            continue
        df = res.merge(weekend_drivers(race.meeting_key), on="driver_number", how="left")
        df["round"] = race.round
        df["meeting_key"] = race.meeting_key
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

quali = load_weekend_session("Qualifying")
quali["position"] = pd.to_numeric(quali["position"], errors="coerce")
quali["best_time"] = quali["duration"].apply(best_time)
quali["gap_to_pole_pct"] = (quali["best_time"] / quali.groupby("round")["best_time"].transform("min") - 1) * 100

sprint = load_weekend_session("Sprint")

print("Kwalifikacje:", quali["round"].nunique(), "weekendów | sprinty:", sprint["round"].nunique(), "weekendów")
quali[quali["meeting_key"] == TARGET_MEETING][["position", "name_acronym", "team_name", "best_time", "gap_to_pole_pct"]].head(10)

Kwalifikacje: 18 weekendów | sprinty: 6 weekendów


,position,name_acronym,team_name,best_time,gap_to_pole_pct
369,1.0,ANT,Mercedes,94.035,0.000000
370,2.0,VER,Red Bull Racing,94.214,0.190355
371,3.0,HAD,Red Bull Racing,94.250,0.228638
372,4.0,LEC,Ferrari,94.198,0.173340
373,5.0,HAM,Ferrari,94.311,0.293508
374,6.0,RUS,Mercedes,94.305,0.287127
375,7.0,PIA,McLaren,94.803,0.816717
376,8.0,NOR,McLaren,94.539,0.535971
377,9.0,LIN,Racing Bulls,94.950,0.973042
378,10.0,LAW,Racing Bulls,95.058,1.087893


## 1.7 Historia Austin (2023–2025)

**Co robimy:** pobieramy kwalifikacje i wyścigi z trzech ostatnich GP USA w Austin i sprawdzamy, jak często wygrywał kierowca z pole position.

**Dlaczego:** w EDA porównamy to z ogólną statystyką sezonu 2026 (pole wygrywało 69% wyścigów). Samochody były inne, ale tor ten sam. OpenF1 ma dane dopiero od 2023, więc próbka to **tylko 3 wyścigi** — traktujemy to jako wskazówkę, nie dowód.

In [8]:
frames = []
for year in [2023, 2024, 2025]:
    at = get("sessions", year=year, location="Austin")
    meeting = at["meeting_key"].iloc[0]
    q_key = at.loc[at["session_name"] == "Qualifying", "session_key"].item()
    r_key = at.loc[at["session_name"] == "Race", "session_key"].item()
    q = get("session_result", session_key=q_key)[["driver_number", "position"]].rename(columns={"position": "quali_pos"})
    r = get("session_result", session_key=r_key)[["driver_number", "position", "dnf"]].rename(columns={"position": "race_pos"})
    df = r.merge(q, on="driver_number", how="left").merge(weekend_drivers(meeting), on="driver_number", how="left")
    df["year"] = year
    frames.append(df)

austin_hist = pd.concat(frames, ignore_index=True)
austin_hist["quali_pos"] = pd.to_numeric(austin_hist["quali_pos"], errors="coerce")
austin_hist["race_pos"] = pd.to_numeric(austin_hist["race_pos"], errors="coerce")

poles = austin_hist[austin_hist["quali_pos"] == 1]
print("Pole position wygrywało wyścig:", (poles["race_pos"] == 1).sum(), "z", len(poles))
austin_hist[austin_hist["race_pos"] <= 3].sort_values(["year", "race_pos"])[["year", "race_pos", "quali_pos", "name_acronym", "team_name"]]

Pole position wygrywało wyścig: 1 z 3


,year,race_pos,quali_pos,name_acronym,team_name
0,2023,1.0,6.0,VER,Red Bull Racing
2,2023,2.0,2.0,NOR,McLaren
3,2023,3.0,4.0,SAI,Ferrari
20,2024,1.0,4.0,LEC,Ferrari
21,2024,2.0,3.0,SAI,Ferrari
22,2024,3.0,2.0,VER,Red Bull Racing
40,2025,1.0,1.0,VER,Red Bull Racing
41,2025,2.0,2.0,NOR,McLaren
42,2025,3.0,3.0,LEC,Ferrari


## 1.8 Najszybsze okrążenie

**Co robimy:** dla każdego zakończonego wyścigu pobieramy wszystkie okrążenia (`laps`) i szukamy najkrótszego.

**Dlaczego:** `session_result` nie mówi, kto miał najszybsze okrążenie — trzeba to policzyć z okrążeń. To nasza czwarta zmienna docelowa.

In [9]:
fl_rows = []
for race in races_done.itertuples():
    laps = get("laps", session_key=race.session_key)
    best = laps.loc[laps["lap_duration"].idxmin()]
    fl_rows.append({"round": race.round, "driver_number": best["driver_number"],
                    "fastest_lap_time": best["lap_duration"], "lap_number": best["lap_number"]})

fastest_laps = pd.DataFrame(fl_rows).merge(
    results[["round", "driver_number", "name_acronym", "team_name", "position"]],
    on=["round", "driver_number"], how="left",
)
fastest_laps

,round,driver_number,fastest_lap_time,lap_number,name_acronym,team_name,position
0,1,3,82.091,42,VER,Red Bull Racing,6.0
1,2,12,95.275,52,ANT,Mercedes,1.0
2,3,12,92.432,49,ANT,Mercedes,1.0
3,4,1,91.869,35,NOR,McLaren,2.0
4,5,12,74.210,68,ANT,Mercedes,1.0
5,6,12,73.481,76,ANT,Mercedes,1.0
6,7,44,80.122,44,HAM,Ferrari,1.0
7,8,12,70.374,59,ANT,Mercedes,3.0
8,9,12,91.777,37,ANT,Mercedes,15.0
9,10,1,108.890,44,NOR,McLaren,7.0


## 1.9 Umiejętności kierowców: porównanie z partnerem z zespołu (2023–2025)

**Co robimy:** dla każdych kwalifikacji z lat 2023–2025 liczymy, o ile procent kierowca był szybszy/wolniejszy od partnera z zespołu.

**Dlaczego:** wynik kierowcy to mieszanka „kierowca + samochód”. Partner z zespołu ma **ten sam samochód**, więc różnica do niego to w dużej mierze umiejętności. Przepisy 2026 zmieniły samochody, ale nie kierowców — dlatego ta informacja przenosi się między epokami, a dane o samochodach (niezawodność, tempo) bierzemy tylko z 2026.

**Jak porównujemy uczciwie:** kierowcy mogą odpaść w różnych segmentach (jeden w Q1, drugi w Q3). Porównujemy czasy z **ostatniego segmentu, w którym jechali obaj** — wtedy warunki toru były takie same.

In [10]:
def teammate_gaps(q):
    """Dla jednej sesji kwalifikacji: różnica % do partnera z ostatniego wspólnego segmentu (Q1/Q2/Q3)."""
    rows = []
    for team, g in q.groupby("team_name"):
        if len(g) != 2:
            continue
        a, b = g.iloc[0], g.iloc[1]
        if not (isinstance(a["duration"], list) and isinstance(b["duration"], list)):
            continue
        common = [i for i in range(3) if a["duration"][i] and b["duration"][i]]
        if not common:
            continue
        k = common[-1]
        gap = (a["duration"][k] / b["duration"][k] - 1) * 100
        rows.append({"name_acronym": a["name_acronym"], "teammate": b["name_acronym"], "gap_pct": gap})
        rows.append({"name_acronym": b["name_acronym"], "teammate": a["name_acronym"], "gap_pct": -gap})
    return pd.DataFrame(rows)

frames = []
for year in [2023, 2024, 2025]:
    q_sessions = get("sessions", year=year, session_name="Qualifying")
    for s in q_sessions.itertuples():
        res = get("session_result", session_key=s.session_key)
        if res.empty:
            continue
        df = res.merge(weekend_drivers(s.meeting_key), on="driver_number", how="left")
        gaps = teammate_gaps(df)
        gaps["year"] = year
        gaps["meeting_key"] = s.meeting_key
        frames.append(gaps)
    print(year, "gotowe")

teammate_hist = pd.concat(frames, ignore_index=True)
print(teammate_hist.shape)
teammate_hist.head()

429 (za dużo zapytań) — czekam 5 s...
429 (za dużo zapytań) — czekam 10 s...
2023 gotowe
429 (za dużo zapytań) — czekam 5 s...
429 (za dużo zapytań) — czekam 10 s...
429 (za dużo zapytań) — czekam 20 s...
429 (za dużo zapytań) — czekam 5 s...
429 (za dużo zapytań) — czekam 10 s...
429 (za dużo zapytań) — czekam 20 s...
2024 gotowe
429 (za dużo zapytań) — czekam 5 s...
429 (za dużo zapytań) — czekam 10 s...
2025 gotowe
(1370, 5)


,name_acronym,teammate,gap_pct,year,meeting_key
0,BOT,ZHO,-0.032797,2023,1141
1,ZHO,BOT,0.032797,2023,1141
2,TSU,DEV,-0.782666,2023,1141
3,DEV,TSU,0.782666,2023,1141
4,OCO,GAS,-0.730085,2023,1141


## 1.10 Zapis

**Co robimy:** zapisujemy wszystkie tabele do CSV.

**Dlaczego:** `02_eda`, `03_model` i `04_prediction` zaczynają od wczytania tych plików — bez dotykania API. Kolumnę `duration` z kwalifikacji (lista) pomijamy, bo CSV nie przechowuje list; potrzebne liczby już z niej wyciągnęliśmy.

In [12]:
Path("data").mkdir(exist_ok=True)
races.to_csv("data/races_2026.csv", index=False)
results.to_csv("data/results_2026.csv", index=False)
quali.drop(columns=["duration", "gap_to_leader"]).to_csv("data/quali_2026.csv", index=False)
sprint.to_csv("data/sprint_2026.csv", index=False)
fastest_laps.to_csv("data/fastest_laps_2026.csv", index=False)
teammate_hist.to_csv("data/teammate_gaps_hist.csv", index=False)
austin_hist.to_csv("data/austin_hist.csv", index=False)
print("Zapisano:", sorted(p.name for p in Path("data").glob("*.csv")))

Zapisano: ['austin_hist.csv', 'fastest_laps_2026.csv', 'quali_2026.csv', 'races_2026.csv', 'results_2026.csv', 'sprint_2026.csv', 'teammate_gaps_hist.csv']
